[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Amaciasagro/GIT-RemoteSensing/blob/master/soils_v2.ipynb)

# 🌱 Soil Spatial Analysis Module 
**Author:** Ariel Macías | Agronomist · GIS & Remote Sensing Data Scientist

**EN:** This notebook performs an advanced spatial analysis of agricultural fields using official data from the USDA-NRCS (Soil Data Mart) and Google Earth Engine (GEE). It extracts, structures, and visualizes edaphic properties for precision agriculture diagnostics.
**ES:** Este módulo realiza un análisis espacial avanzado de lotes agrícolas utilizando datos oficiales de la USDA-NRCS y Google Earth Engine. Extrae, estructura y visualiza propiedades edáficas para el diagnóstico en agricultura de precisión.

### ⚙️ Workflow / Flujo de Trabajo
| Step | Description (EN / ES) |
|------|-----------------------|
| **0** | Configuration & Credentials / *Configuración y Credenciales* |
| **1** | GEE Initialization & AOI Definition / *Inicialización y Definición del Lote* |
| **2** | WFS Download & Textural Mapping / *Descarga WFS y Mapeo Textural* |
| **3** | Tabular SDA Query & Agronomic Report / *Consulta Tabular SDA y Reporte* |
| **4** | Granulometric Composition Charts / *Gráficos de Composición Granulométrica* |

> ⚠️ **Note:** USDA-NRCS data is restricted to the **United States** territory.

In [ ]:
# ============================================================\n
# STEP 0: SETUP & IMPORTS
# ============================================================\n
import os
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']
os.environ['MPLBACKEND'] = 'pdf'
import matplotlib
import matplotlib.colors
import matplotlib.pyplot as plt
import ee
import geemap
import folium 
import geopandas as gpd
import pandas as pd
import requests
import io
import urllib
import warnings
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.express as px
from shapely.geometry import Polygon, mapping
from shapely.ops import unary_union
from IPython.display import display, HTML, FileLink
from contextlib import redirect_stdout
from folium.features import GeoJsonTooltip
import json
import tempfile
import os

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# --- CONFIGURATION VARIABLES ---
GEE_PROJECT  = 'my-project-12126-484118'      # <-- Replace with your GEE Project ID
CENTER_LAT   = 33.584              # Initial map latitude (Texas/US region)
CENTER_LON   = -101.845            # Initial map longitude
INITIAL_ZOOM = 14

print("✅ Setup complete. Environment ready.")

✅ Setup complete. Environment ready.


### 1. Area of Interest (AOI) Definition / Definición del Lote
**EN:** Authenticate Google Earth Engine. Use the drawing tool on the map to define the field boundaries, or upload a spatial file (`.shp` or `.geojson`).
**ES:** Autentica GEE. Utiliza la herramienta de dibujo para delimitar el lote, o sube un archivo espacial.

In [ ]:
# ============================================================\n
# STEP 1: GEE AUTH & AOI DEFINITION
# ============================================================\n

# --- GEE Authentication ---
try:
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine initialized successfully.")
except Exception:
    print("🔑 Authenticating Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine initialized successfully.")

# --- Shared Global Variables ---
field_geom_ee  = None   # ee.Geometry object
field_geom_shp = None   # shapely geometry object

# --- Interactive Map Setup ---
interactive_map = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=INITIAL_ZOOM)
interactive_map.add_basemap('SATELLITE')

# --- UI Widgets: Drawing ---
btn_confirm_draw = widgets.Button(
    description='✅ Confirm Drawn Polygon',
    button_style='success',
    layout=widgets.Layout(width='260px', margin='4px')
)
out_draw = widgets.Output()

def confirm_polygon(b):
    global field_geom_ee, field_geom_shp
    with out_draw:
        out_draw.clear_output()
        roi = interactive_map.user_roi
        if roi is None:
            print("⚠️ No polygon detected. Please draw one first.")
            return

        roi_info       = roi.getInfo()
        field_geom_ee  = ee.Geometry(roi_info.get('geometry', roi_info))
        coords         = field_geom_ee.coordinates().getInfo()[0]
        field_geom_shp = Polygon(coords)

        area_ha = field_geom_shp.area * 111320**2 / 10000
        print(f"✅ Polygon confirmed: {len(coords)-1} vertices")
        print(f"   Estimated Area: {area_ha:.2f} ha")

btn_confirm_draw.on_click(confirm_polygon)

# --- UI Widgets: File Upload ---
uploader = widgets.FileUpload(
    description='📂 Upload Field (.shp / .geojson)',
    accept='.shp,.zip,.geojson',
    multiple=True,
    layout=widgets.Layout(width='260px', margin='4px')
)
btn_upload = widgets.Button(
    description='📌 Load File to Map',
    button_style='info',
    layout=widgets.Layout(width='260px', margin='4px')
)
out_upload = widgets.Output()

def load_spatial_file(b):
    global field_geom_ee, field_geom_shp
    with out_upload:
        out_upload.clear_output()
        files = uploader.value

        if not files:
            print("⚠️ No file uploaded.")
            return

        tmp_dir = tempfile.mkdtemp()
        
        for file in files:
            name, content = file['name'], file['content']
            with open(os.path.join(tmp_dir, name), 'wb') as f:
                f.write(content)

        try:
            # File handling logic (extract zip if needed)
            zip_files = [f for f in os.listdir(tmp_dir) if f.endswith('.zip')]
            if zip_files:
                import zipfile
                with zipfile.ZipFile(os.path.join(tmp_dir, zip_files[0]), 'r') as z:
                    z.extractall(tmp_dir)
            
            valid_files = [f for f in os.listdir(tmp_dir) if f.endswith(('.shp', '.geojson'))]
            if not valid_files:
                print("❌ No valid spatial file found.")
                return

            gdf = gpd.read_file(os.path.join(tmp_dir, valid_files[0])).to_crs(epsg=4326)
            
            field_geom_shp = unary_union(gdf.geometry)
            field_geom_ee  = ee.Geometry(mapping(field_geom_shp))
            
            area_ha = field_geom_shp.area * 111320**2 / 10000
            print(f"✅ File loaded. Estimated Area: {area_ha:.2f} ha")

            with redirect_stdout(io.StringIO()):
                interactive_map.add_gdf(gdf, layer_name="Uploaded Field", style={'color': '#f1c40f', 'fillOpacity': 0.2})
        except Exception as e:
            print(f"❌ Error processing file: {e}")

btn_upload.on_click(load_spatial_file)

# --- Render UI ---
display(widgets.HTML("<h3 style='margin:8px 0'>📍 AOI Definition Panel</h3>"))
display(interactive_map)
display(widgets.HBox([
    widgets.VBox([widgets.HTML("<b>Option A: Draw</b>"), btn_confirm_draw, out_draw]),
    widgets.VBox([widgets.HTML("<b>Option B: Upload</b>"), uploader, btn_upload, out_upload])
]))

✅ Earth Engine initialized successfully.


HTML(value="<h3 style='margin:8px 0'>📍 AOI Definition Panel</h3>")

Map(center=[33.584, -101.845], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

### 2. Spatial WFS Query & Textural Mapping / Consulta WFS y Mapa Textural
**EN:** Connects to the USDA Soil Data Mart WFS service to retrieve spatial mapping units intersecting the AOI. Renders a choropleth map based on dominant soil texture.
**ES:** Se conecta al servicio WFS de USDA para descargar las unidades cartográficas. Genera un mapa coloreado según la textura dominante del suelo.

In [14]:
# ============================================================\n
# STEP 2: WFS DOWNLOAD & CHOROPLETH MAP
# ============================================================\n

if field_geom_ee is None:
    raise ValueError("⚠️ AOI not defined. Please draw or upload a field in Step 1.")

# 1. Bounding Box for WFS Query
minx, miny, maxx, maxy = field_geom_shp.bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}"
centroid = [field_geom_shp.centroid.y, field_geom_shp.centroid.x]

# 2. WFS Request
wfs_url = "https://sdmdataaccess.nrcs.usda.gov/Spatial/SDMNAD83Geographic.wfs"
params  = {
    'SERVICE': 'WFS', 'VERSION': '1.0.0', 'REQUEST': 'GetFeature',
    'TYPENAME': 'MapunitPoly', 'BBOX': bbox_str, 'SRSNAME': 'EPSG:4326', 'OUTPUTFORMAT': 'GML3'
}

print("📡 Connecting to USDA Soil Data Mart (WFS)...")
response = requests.get(wfs_url, params=params, verify=False, timeout=30)

if response.status_code != 200:
    raise ConnectionError("❌ USDA WFS Connection failed.")

gdf_soils = gpd.read_file(io.BytesIO(response.content))
if gdf_soils.empty:
    raise ValueError("⚠️ No soil data found in this area (US coverage only).")

gdf_field   = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[field_geom_shp])
gdf_clipped = gpd.overlay(gdf_soils, gdf_field, how='intersection')
print(f"✅ {len(gdf_clipped)} soil units retrieved and clipped to AOI.")

# 3. Tabular Query for Dominant Texture
mukeys = tuple(int(k) for k in gdf_clipped['mukey'].unique())
mukeys_str = str(mukeys).replace(',)', ')') if len(mukeys) > 1 else f"({mukeys[0]})"

sql_texture = f"""
SELECT c.mukey, c.comppct_r, ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r
FROM component AS c
LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey
WHERE c.mukey IN {mukeys_str} AND c.majcompflag = 'Yes'
AND ch.hzdept_r = (SELECT MIN(hzdept_r) FROM chorizon WHERE chorizon.cokey = c.cokey)
"""

def get_texture_class(sand, silt, clay):
    if any(pd.isna([sand, silt, clay])): return 'No Data'
    s, si, c = float(sand), float(silt), float(clay)
    if c >= 40 and s <= 45 and si <= 40: return 'Clay'
    if c >= 40 and si >= 40:             return 'Silty Clay'
    if c >= 35 and s >= 45:              return 'Sandy Clay'
    if c >= 27 and s <= 45 and si >= 28: return 'Clay Loam'
    if c >= 27 and s >= 45:              return 'Sandy Clay Loam'
    if c >= 27 and si >= 60:             return 'Silty Clay Loam'
    if si >= 50 and c < 27:              return 'Silt Loam'
    if si >= 80 and c < 12:              return 'Silt'
    if s >= 85 and c < 10:               return 'Sand'
    if s >= 70 and c < 15:               return 'Loamy Sand'
    return 'Loam'

TEXTURE_PALETTE = {
    'Clay': '#8B0000', 'Silty Clay': '#B22222', 'Sandy Clay': '#CD5C5C',
    'Clay Loam': '#D2691E', 'Sandy Clay Loam': '#A0522D', 'Silty Clay Loam': '#C68642',
    'Silt Loam': '#DEB887', 'Silt': '#F5DEB3', 'Loam': '#8FBC8F',
    'Loamy Sand': '#F4A460', 'Sand': '#EDC9Af', 'No Data': '#AAAAAA'
}

# Fetch textures
df_texture = pd.DataFrame()
try:
    res = requests.post("https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest",
                        data={'query': sql_texture, 'format': 'JSON'}, verify=False)
    if res.status_code == 200 and 'Table' in res.json():
        df_texture = pd.DataFrame(res.json()['Table'], columns=['mukey','pct','sand','silt','clay'])
        df_texture['texture'] = df_texture.apply(lambda row: get_texture_class(row['sand'], row['silt'], row['clay']), axis=1)
except Exception as e:
    print(f"⚠️ Texture query failed: {e}")

# 4. Render Choropleth Map
soil_map = geemap.Map(center=centroid, zoom=14)
soil_map.add_basemap('SATELLITE')

gdf_render = gdf_clipped[['mukey', 'musym', 'geometry']].copy()
if not df_texture.empty:
    dom_tex = df_texture.sort_values('pct', ascending=False).drop_duplicates('mukey')[['mukey','texture']]
    gdf_render['mukey'] = gdf_render['mukey'].astype(str)
    dom_tex['mukey'] = dom_tex['mukey'].astype(str)
    gdf_render = gdf_render.merge(dom_tex, on='mukey', how='left').fillna({'texture': 'No Data'})
else:
    gdf_render['texture'] = 'No Data'

for texture, group in gdf_render.groupby('texture'):
    color = TEXTURE_PALETTE.get(texture, '#AAAAAA')
    # fillColor sets the polygon fill; color sets the border; both must be specified
    style = {'color': '#555', 'weight': 1, 'fillColor': color, 'fillOpacity': 0.65}
    hover = {'fillColor': color, 'fillOpacity': 0.9}
    with redirect_stdout(io.StringIO()):
        soil_map.add_gdf(group, layer_name=f"Texture: {texture}",
                         style=style, hover_style=hover)

with redirect_stdout(io.StringIO()):
    soil_map.add_gdf(gdf_field, layer_name='AOI Boundary',
                     style={'color': 'white', 'weight': 2.5, 'fillOpacity': 0})

display(soil_map)

📡 Connecting to USDA Soil Data Mart (WFS)...
✅ 20 soil units retrieved and clipped to AOI.


Map(center=[33.674717, -101.6656815], controls=(WidgetControl(options=['position', 'transparent_bg'], position…

### 3. Agronomic Tabular Report / Reporte Agronómico
**EN:** Queries the Soil Data Access (SDA) API to retrieve critical chemical and physical properties (pH, Organic Matter, CEC, AWC) for structural diagnosis.
**ES:** Consulta la API de SDA para extraer propiedades químicas y físicas críticas (pH, Materia Orgánica, CIC, Agua Disponible) para el diagnóstico estructural.

In [20]:
# ============================================================
# STEP 3: AGRONOMIC REPORT WITH UC DAVIS INTEGRATION
# ============================================================
import urllib3, urllib.parse

# EN: Mute SSL warnings / ES: Silenciamos advertencias de SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("\n--- GENERATING SOIL REPORT (SDA + UC DAVIS) ---")

# 1. EN: Area Calculation / ES: Preparar MUKEYs y calcular Hectáreas
gdf_temp = gdf_clipped.copy()
gdf_temp['hectareas'] = gdf_temp.to_crs(epsg=3857).area / 10000

resumen_mukey = gdf_temp.groupby(['mukey', 'musym']).agg({'hectareas': 'sum'}).reset_index().sort_values(by='hectareas', ascending=False)
total_ha = resumen_mukey['hectareas'].sum()
muke_list = resumen_mukey['mukey'].unique().tolist()
mukeys_str = str(tuple(muke_list)).replace(",)", ")")

# 2. EN: SQL Query / ES: SQL para traer componentes y texturas
sql_query = f"""
SELECT c.mukey, c.compname, c.comppct_r, ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r, ch.om_r
FROM component AS c LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey
WHERE c.mukey IN {mukeys_str} AND ch.hzdept_r = (SELECT MIN(hzdept_r) FROM chorizon WHERE chorizon.cokey = c.cokey)
"""

# 3. EN: SDA Request / ES: Petición a SDA
series_data = pd.DataFrame()
try:
    r = requests.post("https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest", data={"query": sql_query, "format": "JSON"}, verify=False, timeout=30)
    if r.status_code == 200 and 'Table' in r.json():
        series_data = pd.DataFrame(r.json()['Table'], columns=['mukey', 'serie', 'pct', 'arena', 'limo', 'arcilla', 'mo'])
        # Convert text numeric columns to float for Plotly compatibility later
        series_data.iloc[:, 2:] = series_data.iloc[:, 2:].apply(pd.to_numeric, errors='coerce')
except Exception as e:
    print(f"❌ USDA Connection Error: {e}")

# 4. EN: HTML Build / ES: Construcción del HTML interactivo
html_output = f"<div style='font-family: sans-serif; max-width: 950px;'><h2 style='color: #2c3e50; border-bottom: 3px solid #2c3e50;'>📊 Soil Properties & Textures Report</h2>"

for _, fila in resumen_mukey.iterrows():
    detalles = series_data[series_data['mukey'].astype(str) == str(fila['mukey'])] if isinstance(series_data, pd.DataFrame) and not series_data.empty else pd.DataFrame()
    pct_lote = (fila['hectareas'] / total_ha) * 100

    html_output += f"""
    <details style="margin-bottom: 12px; border: 1px solid #bdc3c7; border-radius: 6px; overflow: hidden;">
        <summary style="background-color: #2c3e50; padding: 15px; font-weight: bold; cursor: pointer; color: #fff;">
            📍 Unit: {fila['musym']} — {fila['hectareas']:.2f} ha ({pct_lote:.1f}%)
        </summary>
        <div style="padding: 15px; background: #fff;">
            <table style="width: 100%; border-collapse: collapse;">
                <thead>
                    <tr style="background-color: #ecf0f1; color: #2c3e50;">
                        <th style="padding: 10px; text-align: left; border: 1px solid #ddd;">Soil Series</th>
                        <th style="border: 1px solid #ddd;">%</th>
                        <th style="border: 1px solid #ddd;">Sand %</th>
                        <th style="border: 1px solid #ddd;">Silt %</th>
                        <th style="border: 1px solid #ddd;">Clay %</th>
                        <th style="border: 1px solid #ddd; color:#e67e22;">O.M. %</th>
                    </tr>
                </thead>
                <tbody>
    """
    for _, s in detalles.iterrows():
        nombre_s = str(s['serie']).strip()
        nombre_s_url = urllib.parse.quote(nombre_s.lower())
        url_ficha = f"https://casoilresource.lawr.ucdavis.edu/sde/?series={nombre_s_url}#osd"

        html_output += f"""
                <tr style="border-bottom: 1px solid #eee;">
                    <td style="padding: 10px; font-weight: bold; border: 1px solid #ddd;">
                        <a href="{url_ficha}" target="_blank" style="color: #3498db; text-decoration: none;">{nombre_s}</a> 🔗
                    </td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #333; ">{s['pct']:.0f}%</td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #333; ">{s['arena'] if pd.notna(s['arena']) else '-'}</td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #333; ">{s['limo'] if pd.notna(s['limo']) else '-'}</td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #333; ">{s['arcilla'] if pd.notna(s['arcilla']) else '-'}</td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #d35400; font-weight: bold;">{s['mo'] if pd.notna(s['mo']) else '-'}</td>
                </tr>"""
    html_output += "</tbody></table></div></details>"

# 5. Mapa coloreado por clase textural
soil_map = geemap.Map(center=centroid, zoom=14)
soil_map.add_basemap('SATELLITE')

gdf_map = gdf_clipped[['mukey', 'musym', 'geometry']].copy()

if isinstance(df_texture, pd.DataFrame) and not df_texture.empty:
    tex_dom = df_texture.sort_values('pct', ascending=False).drop_duplicates('mukey')[['mukey','texture']]
    # ✅ Fix mukey dtype mismatch
    gdf_map['mukey'] = gdf_map['mukey'].astype(str)
    tex_dom['mukey'] = tex_dom['mukey'].astype(str)
    gdf_map = gdf_map.merge(tex_dom, on='mukey', how='left')
    gdf_map['texture'] = gdf_map['texture'].fillna('No Data')
else:
    gdf_map['texture'] = 'No Data'

for texture, group in gdf_map.groupby('texture'):
    color = TEXTURE_PALETTE.get(texture, '#AAAAAA') # ¡Acá está el problema!
    style = {'color': color, 'weight': 1, 'fillColor': color, 'fillOpacity': 0.55}
    hover = {'fillOpacity': 0.85}
    with redirect_stdout(io.StringIO()):
        soil_map.add_gdf(group, layer_name=f"Textura: {texture}",
                         style=style, hover_style=hover)

with redirect_stdout(io.StringIO()):
    soil_map.add_gdf(gdf_field, layer_name='AOI Boundary / Límite del lote',
                     style={'color': 'white', 'weight': 2, 'fillOpacity': 0})
display(soil_map)



--- GENERATING SOIL REPORT (SDA + UC DAVIS) ---


Map(center=[33.674717, -101.6656815], controls=(WidgetControl(options=['position', 'transparent_bg'], position…

### 4. Granulometric Composition Analysis / Análisis Granulométrico
**EN:** Automated visualization of sand, silt, and clay proportions using Plotly.
**ES:** Visualización automatizada de las proporciones de arena, limo y arcilla utilizando Plotly.

In [21]:
# ============================================================
# STEP 4: PLOTLY GRANULOMETRIC VISUALIZATIONS
# ============================================================

if not series_data.empty:
    # EN: Filter rows with valid granulometric data / ES: Filtrar filas con datos válidos
    plot_data = series_data.dropna(subset=['arena', 'limo', 'arcilla']).copy()
    plot_data = plot_data.sort_values('serie')

    if not plot_data.empty:
        # EN: Create Stacked Bar Chart / ES: Crear gráfico de barras apiladas
        fig = go.Figure()

        fig.add_trace(go.Bar(
            y=plot_data['serie'], x=plot_data['arena'],
            name='Sand / Arena (%)', orientation='h', marker=dict(color='#EDC9Af')
        ))
        fig.add_trace(go.Bar(
            y=plot_data['serie'], x=plot_data['limo'],
            name='Silt / Limo (%)', orientation='h', marker=dict(color='#F5DEB3')
        ))
        fig.add_trace(go.Bar(
            y=plot_data['serie'], x=plot_data['arcilla'],
            name='Clay / Arcilla (%)', orientation='h', marker=dict(color='#8B0000')
        ))

        fig.update_layout(
            title="Granulometric Composition per Soil Series",
            barmode='stack',
            xaxis_title="Composition Percentage (%)",
            yaxis_title="Soil Series",
            plot_bgcolor='white',
            height=400 + (len(plot_data) * 20),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )

        fig.show()
    else:
        print("⚠️ Not enough continuous numerical data to plot granulometry.")
else:
    print("⚠️ No data available to generate charts.")

In [22]:
# ============================================================
# EN: CELL 3+4 — FULL HORIZON QUERY + WEIGHTED AVERAGES
# ES: CELDA 3+4 — CONSULTA COMPLETA DE HORIZONTES + PROMEDIOS PONDERADOS
# Interactive table by MUKey with critical value alerts
# ============================================================
import math
import numpy as np

# ⚠️ EN: Make sure these constants are defined in your Cell 0 / ES: Asegurate de definir estas constantes en tu Celda 0
MAX_DEPTH_CM = 80  # Reemplaza a PROF_MAX_CM
N_FACTOR = 0.05    # Reemplaza a FACTOR_N

# ── EN: Main SQL: all horizons + properties / ES: SQL principal ────
sql_horizons = (
    "SELECT\n"
    "    c.mukey, c.cokey, c.compname, c.comppct_r, c.majcompflag,\n"
    "    ch.chkey, ch.hzname, ch.hzdept_r, ch.hzdepb_r,\n"
    "    ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r,\n"
    "    ch.dbovendry_r,\n"
    "    ch.wtenthbar_r, ch.wthirdbar_r, ch.wfifteenbar_r,\n"
    "    ch.om_r,\n"
    "    ch.ph1to1h2o_r,\n"
    "    ch.ec_r, ch.ecec_r,\n"
    "    ch.esp_r, ch.sar_r,\n"
    "    ch.cec7_r\n"
    "FROM component AS c\n"
    "LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey\n"
    f"WHERE c.mukey IN {mukeys_str}\n"
    "  AND ch.hzdept_r IS NOT NULL\n"
    "  AND ch.hzdepb_r IS NOT NULL\n"
    "ORDER BY c.mukey, c.comppct_r DESC, ch.hzdept_r ASC"
)

HORIZON_COLS = [
    'mukey', 'cokey', 'compname', 'comppct_r', 'majcompflag',
    'chkey', 'hzname', 'hzdept_r', 'hzdepb_r',
    'sandtotal_r', 'silttotal_r', 'claytotal_r',
    'dbovendry_r',
    'wtenthbar_r', 'wthirdbar_r', 'wfifteenbar_r',
    'om_r',
    'ph1to1h2o_r',
    'ec_r', 'ecec_r',
    'esp_r', 'sar_r',
    'cec7_r'
]

NUMERIC_COLS = HORIZON_COLS[3:4] + HORIZON_COLS[7:]

df_horizons = pd.DataFrame()
try:
    print("📡 EN: Querying chorizon table / ES: Consultando tabla chorizon...")
    r = requests.post(
        "https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest",
        data={'query': sql_horizons, 'format': 'JSON'},
        verify=False, timeout=60
    )
    if r.status_code == 200 and 'Table' in r.json():
        df_horizons = pd.DataFrame(r.json()['Table'], columns=HORIZON_COLS)
        for col in NUMERIC_COLS:
            df_horizons[col] = pd.to_numeric(df_horizons[col], errors='coerce')
        df_horizons['n_estimado_r'] = df_horizons['om_r'] * N_FACTOR
        df_horizons['thickness_cm'] = df_horizons['hzdepb_r'] - df_horizons['hzdept_r']
        print(f"✅ EN: {len(df_horizons)} horizons downloaded / ES: {len(df_horizons)} horizontes descargados.")
    else:
        print(f"⚠️ EN: SDA Server returned code / ES: El servidor SDA devolvió código: {r.status_code}")
except Exception as e:
    print(f"❌ EN: SDA Query Error / ES: Error al consultar SDA: {e}")


# ── EN: Helper Functions / ES: Funciones auxiliares ────────────────────────

def calc_weighted_average(df_comp, variable, max_depth=MAX_DEPTH_CM):
    """EN: Weighted avg by root zone depth / ES: Promedio ponderado por espesor en zona radicular."""
    df = df_comp.copy()
    df['top_eff'] = df['hzdept_r'].clip(lower=0, upper=max_depth)
    df['bot_eff'] = df['hzdepb_r'].clip(lower=0, upper=max_depth)
    df['esp_eff'] = df['bot_eff'] - df['top_eff']
    df = df[(df['esp_eff'] > 0) & df[variable].notna()].copy()
    if df.empty: return np.nan
    total_esp = df['esp_eff'].sum()
    return (df[variable] * df['esp_eff']).sum() / total_esp if total_esp > 0 else np.nan

def calc_wpavg_by_mukey(df_horizons, variable, max_depth=MAX_DEPTH_CM):
    """EN: Calculate wpavg by mukey using dominant component / ES: Calcula wpavg por mukey."""
    results = []
    for mukey, df_mu in df_horizons.groupby('mukey'):
        comp_dom = df_mu.sort_values('comppct_r', ascending=False)['cokey'].iloc[0]
        df_comp  = df_mu[df_mu['cokey'] == comp_dom]
        value    = calc_weighted_average(df_comp, variable, max_depth)
        results.append({'mukey': str(mukey), f'{variable}_wpavg': value})
    return pd.DataFrame(results)

def classify_texture(sand, silt, clay):
    if any(v is None or (isinstance(v, float) and np.isnan(v)) for v in [sand, silt, clay]): return 'No Data'
    s, si, c = float(sand), float(silt), float(clay)
    if c >= 40 and s <= 45 and si <= 40: return 'Clay'
    if c >= 40 and si >= 40:             return 'Silty Clay'
    if c >= 35 and s >= 45:              return 'Sandy Clay'
    if c >= 27 and s <= 45 and si >= 28: return 'Clay Loam'
    if c >= 27 and s >= 45:              return 'Sandy Clay Loam'
    if c >= 27 and si >= 60:             return 'Silty Clay Loam'
    if si >= 50 and c < 27:              return 'Silt Loam'
    if si >= 80 and c < 12:              return 'Silt'
    if s >= 85 and c < 10:               return 'Sand'
    if s >= 70 and c < 15:               return 'Loamy Sand'
    return 'Loam'

TEXTURE_PALETTE = {
    'Clay': '#8B0000', 'Silty Clay': '#B22222', 'Sandy Clay': '#CD5C5C',
    'Clay Loam': '#D2691E', 'Sandy Clay Loam': '#A0522D', 'Silty Clay Loam': '#C68642',
    'Silt Loam': '#DEB887', 'Silt': '#F5DEB3', 'Loam': '#8FBC8F',
    'Loamy Sand': '#F4A460', 'Sand': '#EDC9Af', 'No Data': '#AAAAAA',
}

MAP_VARIABLES = {
    'Clay / Arcilla (%)':             'claytotal_r',
    'Sand / Arena (%)':               'sandtotal_r',
    'Silt / Limo (%)':                'silttotal_r',
    'Bulk Dens / Dens. ap. (g/cm³)':  'dbovendry_r',
    'Water / Agua 10 kPa (%)':        'wtenthbar_r',
    'Water / Agua 33 kPa (%)':        'wthirdbar_r',
    'Water / Agua 1500 kPa (%)':      'wfifteenbar_r',
    'OM / Mat. Orgánica (%)':         'om_r',
    'pH (H₂O 1:1)':                   'ph1to1h2o_r',
    'EC / CE (dS/m)':                 'ec_r',
    'ECEC (cmol/kg)':                 'ecec_r',
    'ESP / PSI (%)':                  'esp_r',
    'SAR':                            'sar_r',
    'CEC / CIC (cmol/kg)':            'cec7_r',
    'Est. N / N estimado (%) ⚠️':     'n_estimado_r',
}

# ── EN: Calculate weighted averages / ES: Calcular promedios ponderados ────
gdf_dash = gdf_clipped.copy()
gdf_dash['mukey'] = gdf_dash['mukey'].astype(str)

if not df_horizons.empty:
    wpavg_frames = []
    for label, col in MAP_VARIABLES.items():
        if col in df_horizons.columns:
            wpavg_frames.append(calc_wpavg_by_mukey(df_horizons, col))
    df_wpavg = wpavg_frames[0].copy()
    for df_tmp in wpavg_frames[1:]:
        df_wpavg = df_wpavg.merge(df_tmp, on='mukey', how='outer')

    # EN: Enriched GeoDataFrame / ES: GeoDataFrame enriquecido
    gdf_dash = gdf_dash.merge(df_wpavg, on='mukey', how='left')
    print(f"✅ EN: Weighted averages calculated / ES: Promedios ponderados calculados (0–{MAX_DEPTH_CM} cm).")

# ── EN: Visual Alert Functions / ES: Funciones de alerta visual ─────────────

def get_alert_style(col, val):
    """EN: Returns CSS styles based on alert thresholds / ES: Devuelve estilos CSS."""
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return 'color:#999; text-align:center;'

    CRITICAL = 'background:#ffe4e1; color:#8b0000; font-weight:600; text-align:center; border-radius:3px;'
    WARNING  = 'background:#fff9e6; color:#b8860b; font-weight:600; text-align:center; border-radius:3px;'
    GOOD     = 'background:#f0f9f0; color:#2d6a2d; text-align:center;'
    NORMAL   = 'text-align:center; color:#333;'

    v = float(val)
    if col == 'ec_r':        return CRITICAL if v >= 4 else (WARNING if v >= 2 else NORMAL)
    if col == 'esp_r':       return CRITICAL if v >= 15 else (WARNING if v >= 6 else NORMAL)
    if col == 'sar_r':       return CRITICAL if v >= 13 else (WARNING if v >= 9 else NORMAL)
    if col in ('ph1to1h2o_r', 'phsaturated_r'):
        if v <= 4.5 or v >= 9.0: return CRITICAL
        if v <= 5.5 or v >= 8.5: return WARNING
    if col == 'claytotal_r': return CRITICAL if v >= 50 else (WARNING if v >= 40 else NORMAL)
    if col == 'om_r':
        if v < 0.5: return CRITICAL
        if v < 1.0: return WARNING
        if v >= 3.0: return GOOD
    if col in ('cec7_r', 'ecec_r'): return CRITICAL if v < 6 else (WARNING if v < 10 else NORMAL)
    if col == 'dbovendry_r': return CRITICAL if v >= 1.75 else (WARNING if v >= 1.6 else NORMAL)
    return NORMAL

def format_val(val, decimals=1):
    if val is None or (isinstance(val, float) and math.isnan(val)): return '<span style="color:#ccc;">—</span>'
    return f"{val:.{decimals}f}"

# ── EN: Table Columns Definition / ES: Columnas de la tabla ──────
TABLE_COL_DEFS = [
    ('Hz / Horiz',          'hzname',        '',   0),
    ('Depth / Prof (cm)',   '__depth__',     '',   0),
    ('Sand / Arena %',      'sandtotal_r',   '',   1),
    ('Silt / Limo %',       'silttotal_r',   '',   1),
    ('Clay / Arcilla %',    'claytotal_r',   '🏔️',  1),
    ('Texture / Textura',   '__texture__',   '',   0),
    ('Dens. (g/cm³)',       'dbovendry_r',   '',   2),
    ('H₂O 10kPa (%)',       'wtenthbar_r',   '',   1),
    ('H₂O 33kPa (%)',       'wthirdbar_r',   '',   1),
    ('H₂O 1500kPa (%)',     'wfifteenbar_r', '',   1),
    ('OM / MO (%)',         'om_r',          '🌿',  2),
    ('pH (H₂O 1:1)',        'ph1to1h2o_r',   '⚗️',  2),
    ('EC / CE (dS/m)',      'ec_r',          '💧',  2),
    ('ECEC',                'ecec_r',        '',   1),
    ('ESP / PSI (%)',       'esp_r',         '🧂',  1),
    ('SAR',                 'sar_r',         '🧂',  1),
    ('CEC / CIC',           'cec7_r',        '',   1),
    ('Est. N (%)',          'n_estimado_r',  '',   3),
]

def build_header():
    ths = [f'<th style="padding:7px 9px; background:#f5f5f5; color:#444; border:1px solid #ddd; white-space:nowrap; font-size:11px; font-weight:600; text-align:center;">{badge} {label}</th>' for label, col, badge, _ in TABLE_COL_DEFS]
    return '<tr>' + ''.join(ths) + '</tr>'

def build_data_row(hz, bg):
    tds = []
    texture = classify_texture(hz.get('sandtotal_r'), hz.get('silttotal_r'), hz.get('claytotal_r'))
    for label, col, badge, decs in TABLE_COL_DEFS:
        if col == '__depth__':
            top, bot = hz.get('hzdept_r'), hz.get('hzdepb_r')
            top_s = str(int(top)) if pd.notna(top) else '?'
            bot_s = str(int(bot)) if pd.notna(bot) else '?'
            tds.append(f'<td style="text-align:center; border:1px solid #e8e8e8; padding:5px 7px; white-space:nowrap; color:#555; font-size:11px;">{top_s}–{bot_s}</td>')
        elif col == '__texture__':
            color = TEXTURE_PALETTE.get(texture, '#aaa')
            tds.append(f'<td style="text-align:center; border:1px solid #e8e8e8; padding:5px 7px;"><span style="background:{color}18; color:{color}; border:1px solid {color}50; padding:2px 6px; border-radius:8px; font-size:10px; white-space:nowrap;">{texture}</span></td>')
        elif col == 'hzname':
            tds.append(f'<td style="padding:5px 9px; border:1px solid #e8e8e8; font-weight:600; color:#333; font-size:12px;">{hz.get(col, "—")}</td>')
        else:
            val = hz.get(col)
            style = get_alert_style(col, val) if col else 'text-align:center; color:#333;'
            tds.append(f'<td style="padding:5px 7px; border:1px solid #e8e8e8; font-size:11px; {style}">{format_val(val, decs)}</td>')
    return f'<tr style="background:{bg};">' + ''.join(tds) + '</tr>'

def build_wpavg_row(wp_dict):
    tds = []
    texture_w = classify_texture(wp_dict.get('sandtotal_r_wpavg'), wp_dict.get('silttotal_r_wpavg'), wp_dict.get('claytotal_r_wpavg'))
    for label, col, badge, decs in TABLE_COL_DEFS:
        if col == '__depth__':
            tds.append(f'<td style="text-align:center; border:1px solid #c8d9e8; padding:6px 7px; font-size:10px; color:#567; font-style:italic;">0–{MAX_DEPTH_CM} cm</td>')
        elif col == '__texture__':
            color = TEXTURE_PALETTE.get(texture_w, '#aaa')
            tds.append(f'<td style="text-align:center; border:1px solid #c8d9e8; padding:6px 7px;"><span style="background:{color}18; color:{color}; border:1px solid {color}50; padding:2px 6px; border-radius:8px; font-size:10px;">{texture_w}</span></td>')
        elif col == 'hzname':
            tds.append(f'<td style="padding:6px 9px; border:1px solid #c8d9e8; font-weight:600; color:#456; font-size:11px; white-space:nowrap;">⚖️ Wt. Avg / Prom. (0–{MAX_DEPTH_CM}cm)</td>')
        else:
            wp_col = f'{col}_wpavg' if col and not col.startswith('__') else None
            val = wp_dict.get(wp_col) if wp_col else None
            base_style = get_alert_style(col, val) if col else 'text-align:center; color:#333;'
            base_style = base_style.replace('background:#ffe4e1', 'background:#ffe8e4').replace('background:#fff9e6', 'background:#fffae8').replace('background:#f0f9f0', 'background:#f0fbf0')
            tds.append(f'<td style="padding:6px 7px; border:1px solid #c8d9e8; font-size:11px; {base_style}">{format_val(val, decs)}</td>')
    return ('<tr style="background:#e8f1f8; border-top:2px solid #567;">' + ''.join(tds) + '</tr>')

# ── EN: Mukey mappings / ES: Mapa mukey → musym y área ──────────────────────────────
gdf_areas = gdf_clipped.copy()
gdf_areas['mukey'] = gdf_areas['mukey'].astype(str)
gdf_areas['area_ha'] = gdf_areas.geometry.to_crs(epsg=6933).area / 10_000
musym_map = gdf_areas.groupby('mukey')['musym'].first().to_dict()
area_map  = gdf_areas.groupby('mukey')['area_ha'].sum().to_dict()

# ── EN: HTML Legend / ES: Leyenda de alertas ──────────────────────
HTML_LEGEND = (
    '<div style="font-family:sans-serif; font-size:11px; margin:12px 0 18px; padding:10px 16px; background:#fafafa; border:1px solid #ddd; border-radius:6px; display:flex; flex-wrap:wrap; gap:8px; align-items:center;">'
    '<b style="color:#333; margin-right:5px;">🔔 Alerts / Alertas:</b>'
    '<span style="background:#ffe4e1; color:#8b0000; padding:2px 8px; border-radius:10px; font-weight:600; font-size:10px;">🔴 CRITICAL / CRÍTICO</span>'
    '<span style="background:#fff9e6; color:#b8860b; padding:2px 8px; border-radius:10px; font-weight:600; font-size:10px;">🟡 WARNING / ALERTA</span>'
    '<span style="background:#f0f9f0; color:#2d6a2d; padding:2px 8px; border-radius:10px; font-size:10px;">🟢 OPTIMAL / ÓPTIMO</span>'
    '<span style="color:#666; font-size:10px;">'
    '&nbsp;EC/CE ≥2 Warn / ≥4 Crit'
    '&nbsp;|&nbsp; ESP/PSI ≥6 Warn / ≥15 Crit'
    '&nbsp;|&nbsp; SAR ≥9 Warn / ≥13 Crit'
    '&nbsp;|&nbsp; pH &lt;5.5 or &gt;8.5 Warn'
    '&nbsp;|&nbsp; Clay/Arcilla ≥40% Warn'
    '&nbsp;|&nbsp; OM/MO &lt;1% Warn / &lt;0.5% Crit'
    '&nbsp;|&nbsp; Dens. ≥1.6 Warn / ≥1.75 Crit'
    '</span></div>'
)

# ── EN: Render Tables / ES: Renderizar tablas por MUKey ───────────────
if not df_horizons.empty:
    wpavg_lookup = df_wpavg.set_index('mukey').to_dict('index') if 'df_wpavg' in dir() and not df_wpavg.empty else {}

    html_all = f'<div style="font-family:sans-serif; max-width:100%; overflow-x:auto;">{HTML_LEGEND}'
    html_all += f'<p style="color:#555; font-size:12px; margin:0 0 12px;">📊 <b>{df_horizons["mukey"].nunique()} Map Units (MUKeys)</b>. Dominant component horizons + weighted average 0–{MAX_DEPTH_CM} cm.</p>'

    for mukey, df_mu in df_horizons.groupby('mukey'):
        mukey_str = str(mukey)
        musym   = musym_map.get(mukey_str, mukey_str)
        area_ha = area_map.get(mukey_str, 0)
        
        comp_dom = df_mu.sort_values('comppct_r', ascending=False)['cokey'].iloc[0]
        df_dom   = df_mu[df_mu['cokey'] == comp_dom].sort_values('hzdept_r').reset_index(drop=True)
        compname, comppct, n_hz = df_dom['compname'].iloc[0], df_dom['comppct_r'].iloc[0], len(df_dom)

        html_all += (
            f'<details open style="margin-bottom:16px; border:1px solid #d0d0d0; border-radius:8px; overflow:hidden; box-shadow:0 1px 3px rgba(0,0,0,0.05);">'
            f'<summary style="background:linear-gradient(135deg,#f7f7f7,#e8e8e8); padding:11px 16px; font-weight:600; cursor:pointer; color:#333; font-size:13px; list-style:none; display:flex; flex-wrap:wrap; align-items:center; gap:10px;">'
            f'<span style="background:#fff; padding:3px 9px; border-radius:12px; font-size:12px; border:1px solid #ccc;">MUKey {mukey_str}</span>'
            f'<span>Unit/Unidad: <b>{musym}</b></span><span style="opacity:0.5;">|</span>'
            f'<span>{compname} <span style="opacity:0.6; font-weight:normal;">({comppct:.0f}% dom.)</span></span><span style="opacity:0.5;">|</span>'
            f'<span style="opacity:0.8;">{area_ha:.1f} ha &nbsp;· {n_hz} horiz.</span>'
            f'</summary><div style="padding:12px; background:#fff; overflow-x:auto;">'
            f'<table style="width:100%; border-collapse:collapse; font-size:12px;">'
            f'<thead>{build_header()}</thead><tbody>'
        )

        for i, (_, hz) in enumerate(df_dom.iterrows()):
            html_all += build_data_row(hz, '#fdfdfd' if i % 2 == 0 else '#ffffff')

        html_all += build_wpavg_row(wpavg_lookup.get(mukey_str, {}))
        html_all += '</tbody></table></div></details>'

    display(HTML(html_all + '</div>'))
else:
    print("⚠️ EN: df_horizons is empty. Check SDA connection. / ES: df_horizons vacío. Revisá la conexión al servidor SDA.")

📡 EN: Querying chorizon table / ES: Consultando tabla chorizon...
✅ EN: 161 horizons downloaded / ES: 161 horizontes descargados.
✅ EN: Weighted averages calculated / ES: Promedios ponderados calculados (0–80 cm).


Hz / Horiz,Depth / Prof (cm),Sand / Arena %,Silt / Limo %,🏔️ Clay / Arcilla %,Texture / Textura,Dens. (g/cm³),H₂O 10kPa (%),H₂O 33kPa (%),H₂O 1500kPa (%),🌿 OM / MO (%),⚗️ pH (H₂O 1:1),💧 EC / CE (dS/m),ECEC,🧂 ESP / PSI (%),🧂 SAR,CEC / CIC,Est. N (%)
Ap,0–15,34.7,37.2,28.1,Clay Loam,1.63,—,33.1,20.3,2.50,7.90,1.00,—,0.0,0.0,23.4,0.125
Bt1,15–48,33.6,32.5,33.9,Clay Loam,1.67,—,34.3,22.5,1.50,8.00,1.00,—,0.0,0.0,24.9,0.075
Bt2,48–97,33.6,32.5,33.9,Clay Loam,1.67,—,34.3,22.5,1.50,8.00,1.00,—,0.0,0.0,24.9,0.075
Btk,97–127,38.3,24.0,37.7,Loam,1.69,—,33.2,22.9,0.50,8.20,1.00,—,0.0,0.0,22.9,0.025
Btkk,127–203,40.2,20.3,39.5,Loam,1.64,—,34.0,24.0,0.30,8.30,1.00,—,0.0,0.0,19.2,0.015
⚖️ Wt. Avg / Prom. (0–80cm),0–80 cm,33.8,33.4,32.8,Clay Loam,1.66,—,34.1,22.1,1.69,7.98,1.00,—,0.0,0.0,24.6,0.084
Hz / Horiz,Depth / Prof (cm),Sand / Arena %,Silt / Limo %,🏔️ Clay / Arcilla %,Texture / Textura,Dens. (g/cm³),H₂O 10kPa (%),H₂O 33kPa (%),H₂O 1500kPa (%),🌿 OM / MO (%),⚗️ pH (H₂O 1:1),💧 EC / CE (dS/m),ECEC,🧂 ESP / PSI (%),🧂 SAR,CEC / CIC,Est. N (%)
Ap,0–12,34.7,37.2,28.1,Clay Loam,1.63,—,33.1,20.3,2.50,7.90,1.00,—,0.0,0.0,23.4,0.125
Bt1,12–46,33.6,32.5,33.9,Clay Loam,1.65,—,34.3,22.5,1.50,8.00,1.00,—,0.0,0.0,24.9,0.075
Bt2,46–95,33.6,32.5,33.9,Clay Loam,1.65,—,34.3,22.5,1.50,8.00,1.00,—,0.0,0.0,24.9,0.075


In [23]:
import pandas as pd

def extract_lab_data(pedon_key):
    # EN: Build the exact URL / ES: Armamos la URL exacta
    url = f"https://ncsslabdatamart.sc.egov.usda.gov/rptExecute.aspx?p={pedon_key}&r=1"
    print(f"📡 EN: Extracting data from / ES: Extrayendo datos de: {url}")
    
    try:
        # EN: read_html reads all tables from the webpage and returns a list of DataFrames
        # ES: read_html lee todas las tablas de la página web y devuelve una lista de DataFrames
        tables = pd.read_html(url)
        print(f"✅ EN: Success! Found {len(tables)} tables. / ES: ¡Éxito! Se encontraron {len(tables)} tablas.")
        
        # EN: In the USDA structure, tables are usually in this order:
        # ES: En la estructura de la USDA, las tablas suelen estar en este orden:
        # tables[0] = Site Data and Classification (Pedon)
        # tables[1] = Particle Size Distribution (PSDA) and Texture
        # tables[4] or [5] = Bases, CEC, pH, and Salts (depends on pedon)
        
        # EN: Save the texture table directly as a CSV file
        # ES: Guardamos la tabla de textura como un archivo CSV directamente
        texture_df = tables[1]
        file_name = f"pedon_{pedon_key}_texture.csv"
        texture_df.to_csv(file_name, index=False)
        print(f"💾 EN: Saved automatically as / ES: Guardado automáticamente como: {file_name}")
        
        # EN: Return all tables for exploration / ES: Devolvemos todas las tablas para exploración
        return tables

    except Exception as e:
        print(f"⚠️ EN: Error connecting or reading table / ES: Error al conectar o leer la tabla: {e}")

# EN: Execute the function with your example / ES: Ejecutamos la función con tu ejemplo
all_tables = extract_lab_data(165)

# EN: To display the texture table on screen: / ES: Para ver la tabla de textura en pantalla:
# display(all_tables[1].head())

📡 EN: Extracting data from / ES: Extrayendo datos de: https://ncsslabdatamart.sc.egov.usda.gov/rptExecute.aspx?p=165&r=1
✅ EN: Success! Found 32 tables. / ES: ¡Éxito! Se encontraron 32 tablas.
💾 EN: Saved automatically as / ES: Guardado automáticamente como: pedon_165_texture.csv


In [60]:
# ============================================================
# EN: CELL 4 — (Unified with Cell 3)
# ES: CELDA 4 — (Unificada con Celda 3)
# ============================================================
# EN: All functions and calculations are in Cell 3.
# ES: Todas las funciones y cálculos están en la Celda 3.

print("ℹ️ EN: Cell 4 unified with Cell 3 — full processing available.")
print("ℹ️ ES: Celda 4 unificada con Celda 3 — procesamiento completo disponible.\n")

n_mukeys = df_horizons['mukey'].nunique() if not df_horizons.empty else 0
n_wpavg  = len(df_wpavg) if 'df_wpavg' in dir() else 0

print(f"   df_horizons  : {len(df_horizons)} horizons/horizontes | {n_mukeys} MUKeys")
print(f"   df_wpavg  : {n_wpavg} MUKeys with weighted avg / con promedio ponderado\n")

print(f"   EN: Included variables: Texture, Bulk Density, Water (10/33/1500 kPa),")
print(f"                           OM, pH (H₂O + paste), EC, ECEC, ESP, SAR,")
print(f"                           CEC, Ca, K, N, S, P")
print(f"   ES: Variables incluidas: Textura, Densidad ap., Agua (10/33/1500 kPa),")
print(f"                            MO, pH (H₂O + pasta), CE, ECEC, PSI, SAR,")
print(f"                            CIC, Ca, K, N, S, P")


ℹ️ EN: Cell 4 unified with Cell 3 — full processing available.
ℹ️ ES: Celda 4 unificada con Celda 3 — procesamiento completo disponible.

   df_horizons  : 175 horizons/horizontes | 11 MUKeys
   df_wpavg  : 11 MUKeys with weighted avg / con promedio ponderado

   EN: Included variables: Texture, Bulk Density, Water (10/33/1500 kPa),
                           OM, pH (H₂O + paste), EC, ECEC, ESP, SAR,
                           CEC, Ca, K, N, S, P
   ES: Variables incluidas: Textura, Densidad ap., Agua (10/33/1500 kPa),
                            MO, pH (H₂O + pasta), CE, ECEC, PSI, SAR,
                            CIC, Ca, K, N, S, P


In [24]:
# ============================================================
# EN: CELL 5 — BLOCK 1: INTERACTIVE CHOROPLETH MAP (FOLIUM)
# ES: CELDA 5 — BLOQUE 1: MAPA COROPLETA INTERACTIVO (FOLIUM)
# ============================================================
from folium.features import GeoJsonTooltip
import folium
import branca.colormap as cm
import ipywidgets as widgets
from IPython.display import display, HTML

# ── EN: Variable Selector / ES: Selector de variable ──
# Asume que tu diccionario VARIABLES_MAP ahora se llama MAP_VARIABLES
MAP_OPTIONS = list(MAP_VARIABLES.keys()) + ['⚠️ Phosphorus / Fósforo (P) — N/A in SDA']

var_selector = widgets.Dropdown(
    options=MAP_OPTIONS,
    value=MAP_OPTIONS[0], # Uso el primer elemento por defecto en vez de hardcodear 'Arcilla'
    description='Variable:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='380px')
)

btn_generate = widgets.Button(
    description='🗺️ Generate Map',
    button_style='primary',
    layout=widgets.Layout(width='160px', margin='4px 8px')
)

out_map = widgets.Output()
current_map = None # Variable global para poder guardar el mapa en HTML al final

def generate_map(b=None):
    global current_map
    with out_map:
        out_map.clear_output(wait=True)
        selected_label = var_selector.value

        # ── EN: Special case for Phosphorus / ES: Caso especial para Fósforo ──
        if 'Fósforo' in selected_label or 'Phosphorus' in selected_label:
            display(HTML("""
            <div style="background:#fff3cd; border:2px solid #ffc107; border-radius:10px;
                        padding:24px 30px; max-width:680px; font-family:sans-serif; margin:12px 0;">
                <h3 style="color:#856404; margin:0 0 10px;">⚠️ EN: Phosphorus (P) Not Available / ES: Fósforo (P) No Disponible</h3>
                <p style="color:#555; margin:0 0 8px;">
                    <strong>EN:</strong> The USDA Soil Data Access (SDA) spatial database does not contain available Phosphorus (P) values for any horizon level.<br>
                    <strong>ES:</strong> La base de datos espacial USDA (SDA) no contiene valores de Fósforo disponible (P) para ningún nivel de horizonte.
                </p>
                <p style="color:#555; margin:0 0 8px;">
                    <strong>EN:</strong> P availability depends on specific mineralogy and fertilization history, hence it is excluded from generalized NRCS pedological models.<br>
                    <strong>ES:</strong> El Fósforo depende de la mineralogía específica y del historial de fertilización, por ello no se incluye en los modelos pedológicos.
                </p>
                <p style="color:#856404; font-weight:bold; margin:0;">
                    📋 Action / Acción: Realizar análisis de laboratorio o cargar datos locales.
                </p>
            </div>
            """))
            return

        data_col = MAP_VARIABLES[selected_label]
        col_wpavg = f'{data_col}_wpavg'

        if col_wpavg not in gdf_dash.columns:
            print(f"⚠️ EN: Column '{col_wpavg}' not found. Check previous cell. / ES: Columna '{col_wpavg}' no encontrada.")
            return

        gdf_plot = gdf_dash[['mukey', 'musym', 'geometry', col_wpavg]].copy()
        gdf_plot = gdf_plot.dropna(subset=[col_wpavg])

        if gdf_plot.empty:
            display(HTML("<div style='padding:16px; background:#f8d7da; border-radius:8px; color:#721c24;'>"
                         "⚠️ EN: No data available for this variable. / ES: No hay datos disponibles para esta variable.</div>"))
            return

        vmin = gdf_plot[col_wpavg].min()
        vmax = gdf_plot[col_wpavg].max()

        # ── EN: Create Folium map / ES: Crear mapa Folium ──
        m = folium.Map(
            location=centroid, # Variable actualizada de centro_lote a field_center
            zoom_start=14,
            tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
            attr='Google Satellite'
        )
        folium.TileLayer('CartoDB positron', name='Light Map / Mapa claro').add_to(m)

        # ── EN: Colormap selection / ES: Selección de paleta ──
        if data_col == 'ph1to1h2o_r':
            colormap = cm.LinearColormap(['#e74c3c','#e67e22','#27ae60','#f39c12','#c0392b'],
                                          vmin=vmin, vmax=vmax, caption=selected_label)
        elif data_col in ('ec_r', 'esp_r', 'sar_r'):
            colormap = cm.LinearColormap(['#27ae60','#f39c12','#e74c3c'], vmin=vmin, vmax=vmax, caption=selected_label)
        else:
            colormap = cm.LinearColormap(['#fffde7','#f9a825','#e65100'], vmin=vmin, vmax=vmax, caption=selected_label)

        colormap.add_to(m)

        # ── EN: Special note for estimated N / ES: Nota especial para N estimado ──
        is_estimated_n = data_col == 'n_estimado_r'

        def get_style(feature):
            val = feature['properties'].get(col_wpavg)
            color = colormap(val) if val is not None else '#CCCCCC'
            return {'fillColor': color, 'color': '#333', 'weight': 1.2, 'fillOpacity': 0.72}

        # ── EN: Custom Tooltip / ES: Tooltip personalizado ──
        n_note = " (Est: OM÷20)" if is_estimated_n else ""
        folium.GeoJson(
            gdf_plot.__geo_interface__,
            style_function=get_style,
            highlight_function=lambda x: {'weight': 3, 'fillOpacity': 0.95},
            tooltip=GeoJsonTooltip(
                fields=['musym', col_wpavg],
                aliases=['Unit / Unidad:', f'{selected_label}{n_note}:'],
                localize=True,
                sticky=True,
                style="font-family:sans-serif; font-size:13px;"
            ),
            name=selected_label
        ).add_to(m)

        # ── EN: Field boundary / ES: Contorno del lote ──
        folium.GeoJson(
            gdf_field.__geo_interface__, # Variable actualizada de gdf_lote a gdf_field
            style_function=lambda x: {'color': 'white', 'weight': 2.5, 'fillOpacity': 0},
            name='AOI Boundary / Límite del lote'
        ).add_to(m)

        folium.LayerControl().add_to(m)

        # ── EN: Map Title / ES: Título del mapa ──
        html_title = f"""
        <div style="position:fixed; top:10px; left:50%; transform:translateX(-50%);
                    background:white; padding:8px 18px; border-radius:8px;
                    box-shadow:0 2px 8px rgba(0,0,0,0.25); font-family:sans-serif;
                    font-size:14px; font-weight:bold; z-index:9999; max-width:90%;">
            🗺️ {selected_label} &nbsp;·&nbsp; Wt. Avg / Prom. Ponderado 0–{MAX_DEPTH_CM} cm
            {'&nbsp;<span style="color:#e67e22;font-weight:bold;">⚠️ Est. from OM/MO</span>' if is_estimated_n else ''}
        </div>
        """
        m.get_root().html.add_child(folium.Element(html_title))

        current_map = m
        display(m)

        # ── EN: Summary Table / ES: Tabla resumen ──
        table_col_name = f'{selected_label} (0–{MAX_DEPTH_CM} cm)'
        html_summary = gdf_plot[['musym', col_wpavg]].rename(
            columns={'musym': 'Unit / Unidad', col_wpavg: table_col_name}
        ).style.format({table_col_name: '{:.2f}'}).set_table_styles(
            [{'selector': 'th', 'props': [('background', '#2c3e50'), ('color', 'white'), ('padding', '8px 14px')]},
             {'selector': 'td', 'props': [('padding', '6px 14px'), ('text-align', 'center')]}]
        ).to_html()
        
        display(HTML(f"<div style='margin-top:12px;'>{html_summary}</div>"))

btn_generate.on_click(generate_map)

control_panel = widgets.VBox([
    widgets.HTML("<h3 style='margin:8px 0; font-family:sans-serif;'>🗺️ Block 1 — Choropleth Map / Bloque 1 — Mapa Coropleta</h3>"
                 "<p style='font-family:sans-serif; color:#555; font-size:13px; margin:0 0 10px;'>"
                 "EN: Select a variable to generate the weighted average map.<br>"
                 "ES: Seleccioná la variable y generá el mapa coropleta por promedio ponderado.</p>"),
    widgets.HBox([var_selector, btn_generate])
], layout=widgets.Layout(padding='14px', border='1px solid #bdc3c7', border_radius='8px', margin='8px 0'))

display(control_panel)
display(out_map)
generate_map()  # Generar con variable por defecto al ejecutar la celda

# ── EN: Save HTML map (Bug Fix) / ES: Guardar mapa en HTML (Bug Corregido) ──
def generate_map(b=None):
    global current_map
    with out_map:
        # ... todo el código existente ...
        current_map = m
        display(m)
        
        # Mover el save acá adentro
        m.save('choropleth_map.html')
        print("✅ Map saved as 'choropleth_map.html'")

Output()